In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Read the data
data = pd.read_csv("data/goodreads_book.csv")
# Only keep columns we need
data.head()     

In [ ]:
# Check column names
data.columns

---
### Filter Columns

In [ ]:
# Only keep columns we need
eda_data = data[['Id', 'Name', 'Authors', 'ISBN', 'PublishYear', 'Publisher', 'Language', 'Description']].copy()
eda_data.head()     

In [ ]:
# Find number of smaples and features
eda_data.shape

In [ ]:
# Check data types of features
eda_data.info()

---
# Data Cleaning

### Find missing values and replace with appropriate substitution.

In [ ]:
# Find number of NaN in each column
eda_data.isna().sum()

In [ ]:
eda_data.Language.unique()

---
### Replace missing publisher info with unknown

In [ ]:
eda_data[["Publisher"]] = eda_data[["Publisher"]].fillna("unknown")

In [ ]:
eda_data.isna().sum()

---
### Remove rows with missing description

In [ ]:
# Remove rows having atleast ine NaN in any feature
# eda_data.dropna(axis=0, how="any", inplace=True)
eda_data.dropna(subset=["Description"], inplace=True)

---
### Remove URLs from the description

In [ ]:
list(eda_data.Description[eda_data.Id == 1099555]) #Description with url and html tag

In [ ]:
import re
url_pattern = re.compile(r'https?://\S+|www\.\S+')
def remove_url(text):   
    return re.sub(url_pattern, r'', text)

eda_data.Description = eda_data.Description.apply(remove_url)

---
### Remove html tags from the description.

In [ ]:
html_pattern = re.compile('<[^>]*>')
def clean_html_tags(text):
    return re.sub(html_pattern, r'', text)

eda_data.Description = eda_data.Description.apply(clean_html_tags)

In [ ]:
list(eda_data.Description[eda_data.Id == 1099555])

---
### Remove punctuations from the description

In [ ]:
import string
punctuations = string.punctuation
def remove_punctuations(text):
    return text.translate(str.maketrans('', '', punctuations))

eda_data.Description = eda_data.Description.apply(remove_punctuations)

In [ ]:
eda_data.Description.head(5)

---
### Remove extra spaces from the text columns and convert the lettercase to lower

In [ ]:
eda_data[["Name", "Authors", "Publisher", "Description"]] = pd.concat([eda_data[col].astype(str).str.lower().str.strip() for col in ["Name", "Authors", "Publisher", "Description"]], axis=1)
eda_data.head()

---
### Find book descriptions with very short length and remove them.

In [ ]:
eda_data["length"] = [len(d.split()) for d in eda_data['Description'].tolist()]
eda_data[eda_data.length.isin(range(0,4))][["Id", "Name", "Description", "length"]]\
.sort_values(by=["length"], ascending=True).head(10)

- As we have removed tags we see empty strings in the description, lets first convert them to NaN and then remove them

In [ ]:
# Replace empty strings of description with NaN
eda_data.Description = eda_data.Description.replace(r'^\s*$', np.nan, regex=True)

eda_data[eda_data.length.isin(range(0,4))][["Id", "Name", "Description", "length"]]\
.sort_values(by=["length"], ascending=True).head()

In [ ]:
eda_data.dropna(subset=["Description"], inplace=True)

eda_data[eda_data.length.isin(range(0,4))][["Id", "Name", "Description", "length"]]\
.sort_values(by=["length"], ascending=True).head()

In [ ]:
print(set(eda_data.Description[eda_data.length.isin(range(0,4))]))

In [ ]:
# Drop records with very short description
eda_data.drop(eda_data.index[eda_data.length.isin(range(0,4))], inplace = True)
print(eda_data[eda_data.length.isin(range(0,4))].shape[0])
del eda_data["length"]

---
### Find if there is duplication of rows/features

In [ ]:
# Find number of duplicated rows
eda_data[eda_data.duplicated()==True].shape[0]

In [ ]:
# Find number of duplicated values of each feature
eda_data.shape[0] - eda_data.nunique()

---
### Drop variants of the same book
- We see that some descriptions are repeated this possibly could be due to different versions of the same book. ISBN has missing values. Repeated values in Authors, Publisher can help in book recommedation. 
- We see that same books have different ISBN becuase an ISBN is assigned to each separate edition and variation of a publication. ISBN is like an identity number for each edition, imprint, impression or version of the same book. For example, an e-book, a paperback and a hardcover edition of the same book will each have a different ISBN (except reprintings).
- Only keep variants where Publisher is not null, if Publisher is missing for all the variants then keep the first occurrence and delete rest.

In [ ]:
# Convert unknown to NaN
eda_data["Publisher"] = eda_data.Publisher.replace('unknown',np.nan)

# Find books that have same Name Author, Description and Publisher
eda_data[eda_data.duplicated(subset=["Name", "Authors", "Description"], keep=False)][["Id", "ISBN", "Name", "Authors", "Publisher", "Language", "Description"]].head(10)

In [ ]:
eda_data = eda_data.sort_values(by="Publisher", na_position='last').drop_duplicates(subset=["Name", "Authors", "Description"], keep='first')

- Though we have deleted rows with same name, authors and description, when we find books having duplicated description we observe certain records, this happens due to minor textual changes in the Name of the book and also because ceratin descriptions are repetitive for different books.

In [ ]:
eda_data[eda_data.duplicated(subset=["Description"], keep=False)][["Id", "ISBN", "Name", "Authors", "Publisher", "Language", "Description"]]

---
### Extract book series information from the name of the book
- Remove irrelevant info from the name of the book to improve efficiency of the tokenization.
- Book names with hashtag represent edition of the book in a series

In [ ]:
series_pattern =  "(?:[;]\s*|\(\s*)([^\(;]*\s*#\s*\d+(?:\.?\d+|\\&\d+|-?\d*))"
def get_book_series_info(text):
    series_info = re.findall(series_pattern, text)
    if series_info:
        series_info = " ".join([i.replace(" ", "_") for i in series_info])
        return series_info
    else:
        return np.nan
    

eda_data['BookSeriesInfo'] = eda_data.Name.apply(get_book_series_info)

book_name_cases = ["Sire Lines, Revised Edition (Blood-Horse Classics Library)", 
                   "Lovers and Ladies (Lovers and Ladies, #5&6)",
                   "Oh, Cuan Lejos Llegaras! (Oh, the Places You'll Go!", "Ranma 1/2, Vol. 28 (Ranma ? (US 2nd), #28)", 
                   "Fairy Realm edition, #1-3 (Fairy Realm, #1-3)", "Fairy Realm edition, #1.3 (Fairy Realm, #1.3)", 
                   "Codes: How to Make Them and Break Them (Murderous Maths, # 14)",
                   "The Best Catch In Texas (Men of the West, #10) (Silhouette Special Edition #1814)",
                   "Patriot Games ((Blood-Horse Classics Library)) (Jack Ryan, #1; Jack Ryan Universe, #2)"]

print("\033[1m{:90}\033[0m\033[1m{:5}\033[0m".format("Book Name", "Series Information"))
for name in book_name_cases:
    print("{!s:90}{!s:5}".format(name, get_book_series_info(name)))


- As we are using regex there will be certain exceptions, for example it misses a book name with nested brackets case. The name of the book is "Ranma 1/2, Vol. 28 (Ranma ? (US 2nd), #28)", it should extract series information as `Ranma ? (US 2nd), #28]`, instead, it extracts `[US 2nd), #28]`

In [ ]:
eda_data[["Id","Name", "BookSeriesInfo"]][eda_data.BookSeriesInfo.isna() == False]

In [ ]:
eda_data.isna().sum()

---
### Remove the series info from the name of the book

In [ ]:
series_remove_pattern = re.compile("(?:[\(]\s*[^\(;]*\s*#\s*\d+(?:\.?\d+|\\&\d+|-?\d*)(?:;|\))|\s*[^\(;]*\s*#\s*\d+(?:\.?\d+|\\&\d+|-?\d*)\))")       
def remove_series_info(text):
    return re.sub(series_remove_pattern, r'', text)

print("\033[1m{:80}\033[0m\033[1m{:5}\033[0m".format("Book Name", "Series Information"))
for name in book_name_cases:
    print("{!s:80}{!s:5}".format(name, remove_series_info(name)))

eda_data["Title"]= eda_data["Name"].str.replace(series_remove_pattern, r'').str.strip()

---
### Impute missing language with book title language

In [ ]:
from nltk.classify.textcat import TextCat
tc = TextCat()

def detect_language(text):
    text = " ".join(text.split()[:5])
    if text.isnumeric():
        return 'eng'
    else:
        return tc.guess_language(text).strip()

In [ ]:
import dask.dataframe as dd
import multiprocessing
ddf = dd.from_pandas(eda_data, npartitions=4*multiprocessing.cpu_count()) 
eda_data["Language"] = ddf.map_partitions(lambda df: df.apply(lambda x: detect_language(x['Name']) if pd.isna(x['Language']) else x['Language'], axis=1)).compute() 

In [ ]:
eda_data.isna().sum()

---
### Remove quotes from Publisher column 

In [ ]:
eda_data["Publisher"] = eda_data["Publisher"].str.replace('"','')

---
### Transform book and author names into single token
- Merge first and last name of authors so that two authors with same first or last name are not considered same when the tokenization happens.
- Also merge name of the book so that it is considered as single token during the processing.

In [ ]:
eda_data["Authors"] = eda_data["Authors"].str.strip().str.replace(' ','_')
eda_data["Title"] = eda_data["Title"].str.strip().str.replace(' ','_')
eda_data["Publisher"] = eda_data["Publisher"].str.strip().str.replace(' ','_')
eda_data.head()

---
### Merge all textual information to create summary
- Combine all the book information related tokens such as book series information, authors, publisher, language, publish year into single summary column.

In [ ]:
eda_data["bow"] = eda_data[["BookSeriesInfo", 'Authors', 'Publisher', 'Language']].fillna('').agg(' '.join, axis=1)
# eda_data.drop(["ISBN", "BookSeriesInfo", 'Authors', 'Publisher'], axis = 1, inplace=True)

In [ ]:
eda_data.bow.iloc[8375]

In [ ]:
eda_data

---
### Save Preprocessed Data

In [ ]:
eda_data.to_csv("data/preprocessed.csv", sep=",", index=False)